In [9]:
from jsonschema import validate
import json

schema = json.loads(open("json/user_actions_definitions.schema.json").read())


actions = {
    "action_context" : {
        "screen_height" : 1000,
        "screen_width" : 1000
    },
    "actions" : [{
        "type" : "sleep",
        "seconds" : 2.1
    }]
}

validate(actions, schema)




In [10]:
%set_env AZURE_OPENAI_ENDPOINT=https://swedenopenairesource.openai.azure.com
%set_env AZURE_OPENAI_ENDPOINT_KEY=7e28b9e1a8cf492eabc27b5742da5aab
%set_env AZURE_OPENAI_DEPLOYMENT_NAME=gpt-4o
%set_env AZURE_OPENAI_API_VERSION=2023-03-15-preview


env: AZURE_OPENAI_ENDPOINT=https://swedenopenairesource.openai.azure.com
env: AZURE_OPENAI_ENDPOINT_KEY=7e28b9e1a8cf492eabc27b5742da5aab
env: AZURE_OPENAI_DEPLOYMENT_NAME=gpt-4o
env: AZURE_OPENAI_API_VERSION=2023-03-15-preview


In [11]:
from jinja2 import Environment, FileSystemLoader
env = Environment(loader = FileSystemLoader('templates'))
template = env.get_template('system_prompt_template.jinja')

JSoA = open(".\\json\\user_actions_definitions.schema.json").read()

test_json = [{"type": "image", "path": "screenshot_2024-09-04_12-36-21.770524"}, {"type": "sleep", "seconds": 24.798092}, {"type": "image", "path": "screenshot_2024-09-04_12-36-24.366949"}, {"type": "mouse", "x": 893, "y": 192}, {"type": "sleep", "seconds": 1.603776}, {"type": "image", "path": "screenshot_2024-09-04_12-36-24.366949"}, {"type": "text", "text": "basil"}, {"type": "key", "name": "Key.space"}, {"type": "text", "text": "chicken"}, {"type": "sleep", "seconds": 1.526264}, {"type": "image", "path": "screenshot_2024-09-04_12-36-27.413536"}, {"type": "mouse", "x": 1409, "y": 208}, {"type": "sleep", "seconds": 2.817028}, {"type": "image", "path": "screenshot_2024-09-04_12-36-30.478012"}, {"type": "mouse", "x": 531, "y": 963}, {"type": "sleep", "seconds": 2.079812}, {"type": "image", "path": "screenshot_2024-09-04_12-36-33.605015"}, {"type": "key", "name": "Key.alt_l"}, {"type": "key", "name": "Key.tab"}]

output = template.render(website = "AllRecipes.com",
                         recipe="italian soup",
                         JSoA = JSoA,
                         JAoA = test_json
                         )
print(output)

You are an AI agent tasked with helping a human find a recipe for italian soup on the website AllRecipes.com.  
For reference, JSON array of actions (JAoA) is a list that contains recods of human action (with corresponding screenshots) that a human recently performed on AllRecipes.com in search of italian soup.
Here is the JSON schema of actions (JSoA) that defines the possible set of actions contained in JAoA:
{
    "$id": "http://microsoft.com/cobra_commander/user_actions_definition.schema.json",
    "$schema": "http://microsoft.com/cobra_commander/schema",
    "definitions": {
        "SleepActionSchema" : {
            "type" : "object",
            "description" : "User sleeps for specified number of seconds.",
            "properties" : {
                "type" : {
                    "type" : "string",
                    "enum" : ["sleep"]
                },
                "seconds" : {"type" : "number", "description" : "Number of seconds to sleep"}
            },
            

In [12]:
import os
from openai import AzureOpenAI
    
client = AzureOpenAI(
    api_key=os.getenv("AZURE_OPENAI_ENDPOINT_KEY"),  
    api_version= os.getenv("AZURE_OPENAI_API_VERSION"), #"2024-02-01",
    azure_endpoint = os.getenv("AZURE_OPENAI_ENDPOINT")
    )
    
deployment_name=os.getenv("AZURE_OPENAI_DEPLOYMENT_NAME") #This will correspond to the custom name you chose for your deployment when you deployed a model. Use a gpt-35-turbo-instruct deployment. 

def process_json(json_str):    
    response = client.chat.completions.create(
        model=deployment_name, # model = "deployment_name".
        messages=[
            {"role": "system", "content": "You are a digital assistant tasked to help extract recipe names from json documents.  Find all words that might consitute a recipe name and respond as json array, for example ['italian', 'wedding', 'soup']"},
            {"role": "user", "content": json_str}
        ]
    )

    return response.choices[0].message.content

test_json = [{"type": "image", "path": "screenshot_2024-09-04_12-36-21.770524"}, {"type": "sleep", "seconds": 24.798092}, {"type": "image", "path": "screenshot_2024-09-04_12-36-24.366949"}, {"type": "mouse", "x": 893, "y": 192}, {"type": "sleep", "seconds": 1.603776}, {"type": "image", "path": "screenshot_2024-09-04_12-36-24.366949"}, {"type": "text", "text": "basil"}, {"type": "key", "name": "Key.space"}, {"type": "text", "text": "chicken"}, {"type": "sleep", "seconds": 1.526264}, {"type": "image", "path": "screenshot_2024-09-04_12-36-27.413536"}, {"type": "mouse", "x": 1409, "y": 208}, {"type": "sleep", "seconds": 2.817028}, {"type": "image", "path": "screenshot_2024-09-04_12-36-30.478012"}, {"type": "mouse", "x": 531, "y": 963}, {"type": "sleep", "seconds": 2.079812}, {"type": "image", "path": "screenshot_2024-09-04_12-36-33.605015"}, {"type": "key", "name": "Key.alt_l"}, {"type": "key", "name": "Key.tab"}]

print(process_json(json.dumps(test_json)))



```json
["basil", "chicken"]
```


In [13]:
import json
import sys
session_id = "session_2024-09-04_12-36-00"

keylog_dir = f"C:\\Users\\antonslutsky\\Dev\\azureml-quickstart\\slm-finetuning\\phi3-vision-finetune\\applications\\AllrecipesAgent\\output\\{session_id}\\"
keylog_txt = "keylog.txt"

keylog_txt_aug = "keylog_aug.txt"

keylog_path = keylog_dir + keylog_txt

keylog_aug_path = keylog_dir + keylog_txt_aug



with open(keylog_path, "r") as keylog_file, open(keylog_aug_path, "w") as keylog_aug_file:
    lines = keylog_file.read().split("\n")
    print("Read lines:", len(lines))
    for line in lines:
        try:
            gpt_response = process_json(line)
            print(f"gpt_response: {gpt_response}")
            gpt_json = gpt_response.replace("json", "").replace("```", "").replace("\'", "\"").strip()
            
            print(f"------ gpt_json {gpt_json} -----------")
            gpt_json = json.loads(gpt_json)
            #print(f"gpt_json: {gpt_json}")
            
            if len(gpt_json) > 0:
                target_name = " ".join(gpt_json)
                print(target_name)

                line_js = json.loads(line)

                #training_input = line_js[:-4]
                #training_output = line_js[-4:]

                # output = template.render(website = "AllRecipes.com",
                #                             recipe=target_name,
                #                             JSoA = JSoA,
                #                             JAoA = training_input
                #                             )

                # output = output.replace("\n", " ")
                
                keylog_aug_file.write(f"{target_name}\t{line}\n")

        except json.JSONDecodeError as e:
            print(f"Failed to process line: {line}", e)


Read lines: 90
gpt_response: ```json
["basil", "chicken"]
```
------ gpt_json ["basil", "chicken"] -----------
basil chicken
gpt_response: ```json
["garlic", "pasta"]
```
------ gpt_json ["garlic", "pasta"] -----------
garlic pasta
gpt_response: ```json
["beef", "stroganoff"]
```
------ gpt_json ["beef", "stroganoff"] -----------
beef stroganoff
gpt_response: ```json
["vegan", "curry"]
```
------ gpt_json ["vegan", "curry"] -----------
vegan curry
gpt_response: ```json
["grilled", "salmon"]
```
------ gpt_json ["grilled", "salmon"] -----------
grilled salmon
gpt_response: ```json
["rice", "salad"]
```
------ gpt_json ["rice", "salad"] -----------
rice salad
gpt_response: ```json
["chinese", "chicken"]
```
------ gpt_json ["chinese", "chicken"] -----------
chinese chicken
gpt_response: ```json
["italian", "pasta"]
```
------ gpt_json ["italian", "pasta"] -----------
italian pasta
gpt_response: ```json
["stuffed", "peppers"]
```
------ gpt_json ["stuffed", "peppers"] -----------
stuffed 